# Titanic: Machine Learning from Disaster

## Goal
Predict whether a passenger survived the Titanic disaster (`0` or `1`).


## Approach
- **Model**: XGBoost Classifier
- **Validation**: Stratified K-Fold CV
- **Preprocessing**: Imputation, Feature Engineering, Label Encoding

In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


## 1. Load Data

In [7]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
submission = pd.read_csv('data/gender_submission.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
train.head()

Train shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Preprocessing & Feature Engineering

In [8]:
# Store PassengerId for submission
test_ids = test['PassengerId']

target = 'Survived'

# Combine for consistent processing
train['is_train'] = 1
test['is_train'] = 0
test[target] = np.nan

df = pd.concat([train, test], axis=0).reset_index(drop=True)

# Impute Missing Values
df['Age'] = df['Age'].fillna(df.groupby(['Pclass', 'Sex'])['Age'].transform('median'))
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Feature Engineering: Title
df['Title'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
rare_titles = ['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# Feature Engineering: Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = 1
df['IsAlone'].loc[df['FamilySize'] > 1] = 0

# Drop unused columns
cols_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df = df.drop(columns=cols_to_drop)

# Encoding
cat_cols = ['Sex', 'Embarked', 'Title']
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

# Split back
X = df[df['is_train'] == 1].drop(columns=['is_train', target])
y = df[df['is_train'] == 1][target].astype(int)
X_test = df[df['is_train'] == 0].drop(columns=['is_train', target])

X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,IsAlone
0,3,1,22.0,1,0,7.2500,2,2,2,0
1,1,0,38.0,1,0,71.2833,0,3,2,0
2,3,0,26.0,0,0,7.9250,2,1,1,1
3,1,0,35.0,1,0,53.1000,2,3,2,0
4,3,1,35.0,0,0,8.0500,2,2,1,1


## 3. Model Training (XGBoost)

In [9]:
folds = 5
skf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = []

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'learning_rate': 0.05,
    'max_depth': 4,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1
}

fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    clf = xgb.XGBClassifier(**params, early_stopping_rounds=100)
    
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    val_probs = clf.predict_proba(X_val)[:, 1]
    val_preds_binary = (val_probs > 0.5).astype(int)
    
    acc = accuracy_score(y_val, val_preds_binary)
    fold_accuracies.append(acc)
    print(f"Fold {fold+1} Accuracy: {acc:.5f}")
    
    test_preds.append(clf.predict_proba(X_test)[:, 1])

print(f"\nOverall CV Accuracy: {np.mean(fold_accuracies):.5f}")

Fold 1 Accuracy: 0.86034
Fold 2 Accuracy: 0.84270
Fold 3 Accuracy: 0.80899
Fold 4 Accuracy: 0.82584
Fold 5 Accuracy: 0.83146

Overall CV Accuracy: 0.83386


## 4. Submission

In [10]:
avg_test_probs = np.mean(test_preds, axis=0)
final_preds = (avg_test_probs > 0.5).astype(int)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Survived': final_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")
submission.head()

Submission saved to submission.csv


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0
